In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.features.feature_engineer.min_features import _detect_star_players
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.utils.helpers import *
from src.utils.dataScraper import *
from live import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'DET': ['Kevin Huerter']}

Out Players:
{'ORL': ['Franz Wagner', 'Jonathan Isaac'], 'TOR': ['Immanuel Quickley']}
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 4 teams with confirmed lineups
Updated 1 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])
s25 = _detect_star_players(s25)

s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26 = _detect_star_players(s26)

base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TOP_PLAYER,SECOND_TOP_PLAYER,THIRD_TOP_PLAYER,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,TOP_PLAYER_ACTIVE,SECOND_PLAYER_ACTIVE,THIRD_PLAYER_ACTIVE,name
27669,NaN,NaN,1005,2025-26,1627759,Jaylen Brown,Jaylen,1610612738,BOS,Boston Celtics,42500117,2026-05-02,BOS vs. PHI,L,40.016667,12,27,0.444,3,9,0.333,6,6,1.000,1,8,9,4,3,0,3,1,3,6,33,-16,55.8,0,0,55.0,1,40:01,1,96.2,96.3,96.3,115.7,115.9,115.9,-19.5,-19.5,-19.5,0.235,1.33,11.1,0.019,0.216,0.100,8.3,8.2,0.500,0.557,0.344,0.354,98.53,98.36,81.97,98.36,0.168,82,12.0,27.0,F,4.00,2.86,9.0,11.0,19.0,85.0,1.0,0.0,49.0,5.0,12.0,0.417,7.0,15.0,0.467,2.0,8.0,0.25,37,93,0.398,13,49,0.265,13,16,0.813,10,34,44,19,5.0,4,7,1,23,19,100,-9.0,105.2,105.3,111.1,116.0,-5.9,-10.7,0.514,3.80,15.2,0.193,0.800,0.461,0.053,0.468,0.500,96.6,94.5,78.75,95,0.433,1610612755,PHI,Philadelphia 76ers,39,82,0.476,11,28,0.393,20,23,0.870,3,41,44,25,9.0,0,1,7,19,23,109,9.0,111.1,116.0,105.2,105.3,5.9,10.7,0.641,2.78,19.5,0.200,0.807,0.539,0.096,0.543,0.592,96.6,94.5,78.75,94,0.567,1,SF,29.0,NaN,NaN,-4.0,205.0,1,0.824656,0.099958,0.224906,1,3,Jaylen Brown,Jayson Tatum,Neemias Queta,1,1,2,1,1,0,1,Jaylen Brown
27670,NaN,NaN,1004,2025-26,202331,Paul George,Paul,1610612755,PHI,Philadelphia 76ers,42500117,2026-05-02,PHI @ BOS,W,41.833333,5,10,0.500,3,5,0.600,0,0,0.000,0,3,3,1,3,0,0,1,4,1,13,9,15.1,0,0,20.0,1,41:50,1,113.5,119.5,119.5,105.9,106.0,106.0,7.6,13.6,13.6,0.033,0.33,7.1,0.000,0.061,0.034,21.4,21.4,0.650,0.650,0.144,0.147,97.76,95.24,79.36,95.24,0.030,82,5.0,10.0,F,3.71,2.80,0.0,7.0,7.0,66.0,1.0,0.0,51.0,3.0,5.0,0.600,2.0,5.0,0.400,1.0,4.0,0.25,39,82,0.476,11,28,0.393,20,23,0.870,3,41,44,25,9.0,0,1,7,19,23,109,9.0,111.1,116.0,105.2,105.3,5.9,10.7,0.641,2.78,19.5,0.200,0.807,0.539,0.096,0.543,0.592,96.6,94.5,78.75,94,0.567,1610612738,BOS,Boston Celtics,37,93,0.398,13,49,0.265,13,16,0.813,10,34,44,19,5.0,4,7,1,23,19,100,-9.0,105.2,105.3,111.1,116.0,-5.9,-10.7,0.514,3.80,15.2,0.193,0.800,0.461,0.053,0.468,0.500,96.6,94.5,78.75,95,0.433,1,PF,35.0,NaN,NaN,4.0,205.0,1,0.310757,0.023904,0.071713,0,1,Joel Embiid,Tyrese Maxey,Paul George,1,0,3,1,1,1,1,Paul G

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_odds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_odds = pd.json_normalize(data)

print("Loaded:", file.name)
team_odds.head()

Loaded: NBA_20260503_112034.json


,home_team,away_team,commence_time,bookmakers
0,Detroit Pistons,Orlando Magic,2026-05-03 19:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Cleveland Cavaliers,Toronto Raptors,2026-05-03 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,New York Knicks,Philadelphia 76ers,2026-05-05 00:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,San Antonio Spurs,Minnesota Timberwolves,2026-05-05 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,Oklahoma City Thunder,Los Angeles Lakers,2026-05-06 01:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = s26
ast_df = s26
reb_df = s26
min_df = s26


#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
# lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == current_date]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
# lines_us = lines_us[lines_us['COMMENCE_TIME'] == current_date]
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-05-03 11:20:34
US latest pull: 2026-05-03 11:19:54


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Cade Cunningham,Over,29.0,-137,2026-05-03,2026-05-03T18:19:48Z,2026-05-03 11:20:34
1,PrizePicks,player_points,Cade Cunningham,Under,29.0,-137,2026-05-03,2026-05-03T18:19:48Z,2026-05-03 11:20:34
2,PrizePicks,player_points,Paolo Banchero,Over,23.5,-137,2026-05-03,2026-05-03T18:19:48Z,2026-05-03 11:20:34
3,PrizePicks,player_points,Paolo Banchero,Under,23.5,-137,2026-05-03,2026-05-03T18:19:48Z,2026-05-03 11:20:34
4,PrizePicks,player_points,Desmond Bane,Over,19.5,-137,2026-05-03,2026-05-03T18:19:48Z,2026-05-03 11:20:34


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-01-16.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-01-01.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-01-01.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-01-01.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

Team odds: NBA_20260503_112034.json
[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Cade Cunningham,PTS,31.52,39.71,43.59,0.3938,0.6288,0.9503,12.41,24.97,41.42,"[0.7396870554765292, 0.8129032258064518, 0.281..."
1,Paolo Banchero,PTS,32.47,39.76,43.00,0.3844,0.5875,0.9088,12.48,23.36,39.08,"[0.3006681514476614, 0.5114401076716016, 0.418..."
2,Desmond Bane,PTS,29.16,36.41,40.95,0.3002,0.5124,0.7897,8.75,18.66,32.34,"[0.6411062225015713, 0.562751228226887, 0.4705..."
3,Tobias Harris,PTS,25.61,34.39,40.13,0.2205,0.4728,0.7585,5.65,16.26,30.44,"[0.620884289746002, 0.4630225080385852, 0.6424..."
4,Jalen Suggs,PTS,27.30,35.37,40.77,0.2046,0.4452,0.7320,5.59,15.75,29.84,"[0.5016077170418006, 0.5305039787798408, 0.269..."
5,Jalen Duren,PTS,23.34,31.23,36.75,0.2926,0.5087,0.8194,6.83,15.88,30.11,"[0.6183115338882283, 0.9544008483563096, 0.344..."
6,Anthony Black,PTS,18.08,26.67,34.37,0.1818,0.4262,0.7321,3.29,11.37,25.16,"[0.3024054982817869, 0.2624384909786769, 0.346..."
7,Duncan Robinson,PTS,23.27,31.47,36.95,0.1701,0.4361,0.7306,3.96,13.72,27.00,"[0.4343105320304017, 0.5335844318895167, 0.412..."
8,Ausar Thompson,PTS,24.72,33.42,39.02,0.1289,0.4118,0.7080,3.19,13.76,27.62,"[0.2498265093684941, 0.3102378490175801, 0.191..."
9,Daniss Jenkins,PTS,8.60,18.97,27.61,0.1630,0.4444,0.7437,1.40,8.43,20.53,"[0.4712990936555892, 0.4136219495381221, 0.574..."


In [9]:
from live import adjust_predictions

# Build contexts dict once (using the notebook's get_game_context)
game_contexts = {
    name: get_game_context(base_df, name, team_odds, is_playoff=True)
    for name in pts_preds["PLAYER_NAME"]
}

# Adjust the model's Q50 predictions with scenario signals
pts_preds = adjust_predictions(pts_preds, base_df, game_contexts)
ast_preds = adjust_predictions(ast_preds, base_df, game_contexts)
reb_preds = adjust_predictions(reb_preds, base_df, game_contexts)
ast_preds.head()

Cade Cunningham [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 40.6→42.0 (Δ+1.41)  RATE: 0.5978→0.5699 (Δ-0.0279)
Paolo Banchero [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 40.3→41.6 (Δ+1.39)  RATE: 0.5619→0.5434 (Δ-0.0185)
Desmond Bane [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 36.5→36.7 (Δ+0.22)  RATE: 0.4631→0.3979 (Δ-0.0652)
Tobias Harris [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 35.5→37.7 (Δ+2.14)  RATE: 0.5264→0.5850 (Δ+0.0586)
Jalen Suggs [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 37.3→40.0 (Δ+2.71)  RATE: 0.4515→0.4512 (Δ-0.0003)
Jalen Duren [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 32.4→34.0 (Δ+1.66)  RATE: 0.4833→0.4250 (Δ-0.0583)
Anthony Black [PTS] [ix: away_underdog]  pace_bucket=low_pace  MIN: 24.2→20.9 (Δ-3.24)  RATE: 0.3911→0.3617 (Δ-0.0294)
Duncan Robinson [PTS] [ix: home_favorite]  pace_bucket=low_pace  MIN: 30.8→30.0 (Δ-0.77)  RATE: 0.4590→0.4717 (Δ+0.0126)
Ausar Thompson [PTS] [ix: home_favorite]  pace_b

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY,ADJ_CONTEXT_OK,ADJ_CONTEXT_ERR,ADJ_ACTIVE_STARS,ADJ_STARS_MISSING,ADJ_SPREAD_ROLE,ADJ_CTX_SPREAD,ADJ_CTX_TOTAL,ADJ_MIN_DELTA,ADJ_RATE_DELTA,ADJ_MIN_SHIFT,ADJ_RATE_SHIFT,ADJ_USED_INTERACTION,ADJ_MIN_LOG,ADJ_RATE_LOG
0,Cade Cunningham,AST,33.82,42.01,45.89,0.0780,0.1613,0.2859,2.64,6.78,13.12,"[0.265895, 0.562045, 0.439415, 0.478638, 0.223...",True,None,3,0,favorite,-8.5,201.5,1.4064,-0.01685,1.41,-0.0169,True,"{'stars': (-0.0316, 88), 'pace': (0.8376, 46),...","{'stars': (-0.0075, 88), 'pace': (-0.0159, 46)..."
1,Paolo Banchero,AST,34.36,41.65,44.89,0.0610,0.1261,0.2149,2.10,5.25,9.65,"[0.089923, 0.205043, 0.103768, 0.150472, 0.147...",True,None,3,0,underdog,8.5,201.5,1.3851,-0.00540,1.39,-0.0054,True,"{'stars': (-0.1202, 31), 'pace': (0.2692, 40),...","{'stars': (-0.0157, 31), 'pace': (0.0002, 40),..."
2,Jalen Suggs,AST,31.95,40.02,45.42,0.0238,0.1060,0.1987,0.76,4.24,9.02,"[0.135976, 0.128726, 0.289426, 0.231204, 0.181...",True,None,3,0,underdog,8.5,201.5,2.7079,-0.03135,2.71,-0.0314,True,"{'stars': (-1.7457, 26), 'pace': (1.8213, 31),...","{'stars': (0.0108, 26), 'pace': (-0.039, 31), ..."
3,Scottie Barnes,AST,33.01,40.03,46.10,0.1694,0.2579,0.3905,5.59,10.32,18.00,"[0.59819, 0.502352, 0.352552, 0.327095, 0.3187...",True,None,3,0,underdog,8.5,209.5,0.9112,0.03770,0.91,0.0377,True,"{'stars': (-0.4518, 69), 'pace': (1.0336, 49),...","{'stars': (-0.0003, 69), 'pace': (0.0008, 49),..."
4,Jamal Shead,AST,26.20,35.17,42.25,0.0684,0.1527,0.2717,1.79,5.37,11.48,"[0.40306, 0.385123, 0.163864, 0.233535, 0.1699...",True,None,3,0,underdog,8.5,209.5,2.6257,-0.01605,2.63,-0.0161,True,"{'stars': (-0.5707, 63), 'pace': (1.2964, 47),...","{'stars': (-0.0004, 63), 'pace': (-0.0097, 47)..."


### Get Line Probabilities

In [10]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
124,Victor Wembanyama,PTS,27.5,20.00,28.48,35.14,13.68,26.22,44.00,0.614,0.386
106,Jaylon Tyson,PTS,6.5,0.27,7.60,14.58,0.04,3.05,10.17,0.184,0.816
126,Ayo Dosunmu,PTS,17.5,24.52,31.32,39.72,6.00,15.29,29.13,0.499,0.500
3,Scottie Barnes,AST,8.0,32.28,39.30,45.37,5.02,9.59,17.09,0.785,0.214
85,Desmond Bane,PTS,18.5,29.30,36.55,41.09,5.91,15.13,28.40,0.336,0.664
41,James Harden,REB,4.5,31.53,39.00,43.38,2.07,5.29,10.52,0.637,0.363
99,Evan Mobley,PTS,16.5,27.67,35.76,41.25,9.30,19.42,32.69,0.726,0.274
117,Paul George,PTS,15.5,32.05,40.40,44.79,7.61,18.19,31.95,0.719,0.281
129,Jaden McDaniels,PTS,16.5,25.82,36.28,42.81,5.93,16.04,31.50,0.581,0.419
33,Jalen Suggs,REB,4.0,31.18,39.25,44.65,1.52,4.81,10.19,0.594,0.406


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
underdog_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
underdog_all_lines

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
7,Tyrese Maxey,AST,6.5,34.29,41.65,44.67,2.49,6.11,10.67,0.380,0.620,AST,Underdog,New York Knicks,7.5,212.5,112.3,7.0,97.71,25.0,106.0,-120.0,0.485,0.545,5.8,5.5,2.25,-0.7,-1.0,0.311,0.378,0.622,-22.13,14.03,0.2,0.3,0.47,0.46,37.96,4.71,0.27,0.05,5.33,6.0
8,Jalen Brunson,AST,6.5,29.23,36.76,41.59,2.93,6.90,13.11,0.630,0.370,AST,Underdog,Philadelphia 76ers,-7.5,212.5,114.4,17.0,100.40,15.0,-106.0,100.0,0.515,0.500,7.2,7.5,3.43,0.7,1.0,-0.204,0.581,0.419,12.91,-16.20,0.6,0.7,0.67,0.56,34.65,3.81,0.31,0.06,5.57,7.0
10,Josh Hart,AST,4.5,22.89,31.01,37.59,0.00,1.98,5.12,0.078,0.922,AST,Underdog,Philadelphia 76ers,-7.5,212.5,114.4,17.0,100.40,15.0,-104.0,-114.0,0.510,0.533,3.8,3.5,1.48,-0.7,-1.0,0.473,0.318,0.682,-37.62,28.02,0.4,0.3,0.27,0.59,32.41,4.11,0.15,0.06,7.71,7.0
19,LeBron James,AST,7.5,27.63,36.26,41.84,2.91,6.70,13.05,0.578,0.422,AST,Underdog,Oklahoma City Thunder,15.8,213.5,106.5,1.0,100.37,16.0,-110.0,-110.0,0.524,0.524,9.4,8.5,3.17,1.9,1.0,-0.599,0.725,0.275,38.41,-47.50,0.4,0.6,0.60,0.52,35.16,7.68,0.32,0.06,5.60,5.0
27,Jalen Duren,REB,9.5,25.76,33.65,39.17,3.47,8.43,15.70,0.350,0.650,REB,Underdog,Orlando Magic,-8.5,202.0,113.6,13.0,100.56,14.0,105.0,-119.0,0.488,0.543,8.5,9.0,0.85,-1.0,-0.5,1.176,0.120,0.880,-75.40,61.95,0.0,0.0,0.27,0.56,29.38,3.32,0.18,0.04,9.08,13.0
28,Paolo Banchero,REB,8.5,33.47,40.76,44.00,5.21,9.99,15.93,0.777,0.223,REB,Underdog,Detroit Pistons,8.5,202.0,108.9,2.0,99.88,19.0,100.0,-105.0,0.500,0.512,8.6,9.0,2.01,0.1,0.5,-0.050,0.520,0.480,4.00,-6.29,0.6,0.6,0.53,0.43,36.29,4.81,0.29,0.06,9.00,10.0
29,Ausar Thompson,REB,7.5,25.32,34.02,39.62,3.40,7.71,13.13,0.735,0.265,REB,Underdog,Orlando Magic,-8.5,202.0,113.6,13.0,100.56,14.0,-109.0,108.0,0.522,0.481,7.4,7.5,3.50,-0.1,0.0,0.029,0.488,0.512,-6.43,6.50,1.0,0.5,0.40,0.23,29.75,5.73,0.14,0.04,8.25,12.0
35,Duncan Robinson,REB,2.5,21.75,29.95,35.43,0.81,3.45,7.74,0.610,0.390,REB,Underdog,Orlando Magic,-8.5,202.0,113.6,13.0,100.56,14.0,105.0,-120.0,0.488,0.545,2.3,2.0,2.00,-0.2,-0.5,0.100,0.460,0.540,-5.70,-1.00,0.6,0.4,0.40,0.45,27.13,4.38,0.16,0.04,2.62,13.0
37,Jarrett Allen,REB,7.5,21.92,29.35,34.59,3.47,8.08,14.68,0.482,0.518,REB,Underdog,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,100.0,100.0,0.500,0.500,7.3,7.0,4.14,-0.2,-0.5,0.048,0.481,0.519,-3.80,3.80,0.2,0.4,0.47,0.65,26.24,3.93,0.18,0.07,8.36,11.0
39,Collin Murray-Boyles,REB,6.5,18.19,27.36,35.91,1.63,5.05,11.88,0.423,0.577,REB,Underdog,Cleveland Cavaliers,8.5,210.5,114.1,15.0,100.70,13.0,-114.0,105.0,0.533,0.488,6.4,7.0,2.37,-0.1,0.5,0.042,0.483,0.517,-9.33,5.98,0.8,0.6,0.47,0.30,24.85,6.89,0.17,0.07,5.88,8.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
prizePicks_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
12,Stephon Castle,AST,7.0,26.86,35.46,41.20,5.41,10.09,16.59,0.904,0.096,AST,PrizePicks,Minnesota Timberwolves,-14.0,216.5,112.5,8.0,101.50,10.0,-137.0,-137.0,0.578,0.578,7.7,7.5,2.83,0.7,0.5,-0.247,0.598,0.402,3.45,-30.46,0.2,0.5,0.60,0.27,33.03,3.97,0.25,0.04,4.00,6.0
84,Paolo Banchero,PTS,23.5,33.47,40.76,44.00,11.15,21.86,37.73,0.407,0.593,PTS,PrizePicks,Detroit Pistons,8.5,202.0,108.9,2.0,99.88,19.0,-105.0,-110.0,0.512,0.524,23.4,21.5,8.98,-0.1,-2.0,0.011,0.496,0.504,-3.16,-3.78,0.4,0.3,0.20,0.50,36.29,4.81,0.29,0.06,25.70,10.0
6,Ja'Kobe Walter,AST,1.5,27.32,39.90,47.20,0.00,0.61,5.93,0.405,0.595,AST,PrizePicks,Cleveland Cavaliers,8.5,210.5,114.1,15.0,100.70,13.0,-109.0,110.0,0.522,0.476,1.5,1.5,1.08,0.0,0.0,0.000,0.500,0.500,-4.13,5.00,0.4,0.5,0.47,0.38,29.17,5.70,0.14,0.04,1.27,11.0
145,Marcus Smart,PTS,10.5,26.79,35.45,41.30,3.64,14.28,28.67,0.579,0.421,PTS,PrizePicks,Oklahoma City Thunder,15.8,213.5,106.5,1.0,100.37,16.0,-106.0,-114.0,0.515,0.533,11.4,10.0,7.28,0.9,-0.5,-0.124,0.549,0.451,6.69,-15.34,0.6,0.5,0.47,0.39,31.33,6.19,0.18,0.06,14.00,2.0
110,Dean Wade,PTS,4.5,14.30,21.46,27.94,1.55,8.00,19.05,0.730,0.270,PTS,PrizePicks,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-125.0,105.0,0.556,0.488,5.7,6.0,2.54,1.2,1.5,-0.472,0.682,0.318,22.76,-34.81,0.8,0.7,0.60,0.56,23.28,3.82,0.10,0.05,5.40,10.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
betr_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
138,Austin Reaves,PTS,21.5,28.57,37.54,43.18,6.54,17.53,33.30,0.303,0.697,PTS,Betr DFS,Oklahoma City Thunder,15.8,213.5,106.5,1.0,100.37,16.0,-111.0,-103.0,0.526,0.507,20.9,20.5,4.28,-0.6,-1.0,0.140,0.444,0.556,-15.60,9.58,0.2,0.5,0.53,0.43,35.26,5.37,0.25,0.05,16.50,6.0
135,Rudy Gobert,PTS,8.5,23.79,32.84,38.91,1.31,10.95,23.95,0.375,0.625,PTS,Betr DFS,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,-114.0,-105.0,0.533,0.512,7.4,5.5,4.72,-1.1,-3.0,0.233,0.408,0.592,-23.41,15.58,0.4,0.4,0.60,0.64,32.82,4.90,0.11,0.04,10.50,6.0
3,Scottie Barnes,AST,8.0,32.28,39.30,45.37,5.02,9.59,17.09,0.785,0.214,AST,Betr DFS,Cleveland Cavaliers,8.5,210.5,114.1,15.0,100.70,13.0,-137.0,-137.0,0.578,0.578,7.8,6.5,3.91,-0.2,-1.5,0.051,0.480,0.520,-16.96,-10.04,0.6,0.4,0.47,0.17,35.67,6.59,0.24,0.06,7.69,13.0
131,Naz Reid,PTS,12.5,17.95,24.98,31.58,2.60,10.20,21.20,0.340,0.660,PTS,Betr DFS,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,102.0,-120.0,0.495,0.545,11.5,12.0,4.72,-1.0,-0.5,0.212,0.416,0.584,-15.97,7.07,0.4,0.4,0.40,0.52,24.88,4.94,0.21,0.03,11.00,7.0
77,Josh Hart,REB,8.5,22.89,31.01,37.59,3.30,7.07,12.73,0.362,0.638,REB,Betr DFS,Philadelphia 76ers,-7.5,212.5,114.4,17.0,100.40,15.0,110.0,-125.0,0.476,0.556,7.6,7.0,4.25,-0.9,-1.5,0.212,0.416,0.584,-12.64,5.12,0.6,0.5,0.33,0.49,32.41,4.11,0.15,0.06,11.43,7.0


In [14]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
draftKings_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
104,Max Strus,PTS,8.5,20.11,25.25,33.67,1.73,9.57,22.90,0.475,0.525,PTS,DraftKings Pick6,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-101.0,-110.0,0.502,0.524,8.4,7.0,6.80,-0.1,-1.5,0.015,0.494,0.506,-1.69,-3.40,0.2,0.3,0.47,0.57,23.75,3.99,0.16,0.05,10.75,8.0
27,Jalen Duren,REB,9.5,25.76,33.65,39.17,3.47,8.43,15.70,0.350,0.650,REB,DraftKings Pick6,Orlando Magic,-8.5,202.0,113.6,13.0,100.56,14.0,105.0,-119.0,0.488,0.543,8.5,9.0,0.85,-1.0,-0.5,1.176,0.120,0.880,-75.40,61.95,0.0,0.0,0.27,0.56,29.38,3.32,0.18,0.04,9.08,13.0
29,Ausar Thompson,REB,7.5,25.32,34.02,39.62,3.40,7.71,13.13,0.735,0.265,REB,DraftKings Pick6,Orlando Magic,-8.5,202.0,113.6,13.0,100.56,14.0,-109.0,108.0,0.522,0.481,7.4,7.5,3.50,-0.1,0.0,0.029,0.488,0.512,-6.43,6.50,1.0,0.5,0.40,0.23,29.75,5.73,0.14,0.04,8.25,12.0
43,Donovan Mitchell,REB,4.5,29.91,36.99,42.11,2.05,4.76,10.10,0.766,0.234,REB,DraftKings Pick6,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,134.0,-135.0,0.427,0.574,5.3,5.5,1.42,0.8,1.0,-0.563,0.713,0.287,66.84,-50.04,1.0,0.8,0.67,0.46,33.63,4.48,0.30,0.06,4.55,11.0
100,Collin Murray-Boyles,PTS,12.5,18.19,27.36,35.91,4.30,14.51,28.78,0.713,0.287,PTS,DraftKings Pick6,Cleveland Cavaliers,8.5,210.5,114.1,15.0,100.70,13.0,-110.0,-103.0,0.524,0.507,13.0,14.5,6.06,0.5,2.0,-0.083,0.533,0.467,1.75,-7.96,0.8,0.6,0.60,0.25,24.85,6.89,0.17,0.07,12.88,8.0


In [15]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
125,Julius Randle,PTS,21.5,25.88,34.68,40.25,9.49,20.35,34.49,0.488,0.512,PTS,PrizePicks,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,109.0,-122.0,0.478,0.550,20.8,20.0,4.89,-0.7,-1.5,0.143,0.443,0.557,-7.41,1.36,0.4,0.4,0.40,0.41,33.35,2.97,0.27,0.04,16.57,7.0
72,Austin Reaves,REB,4.5,28.57,37.54,43.18,2.19,5.17,10.87,0.679,0.321,REB,Betr DFS,Oklahoma City Thunder,15.8,213.5,106.5,1.0,100.37,16.0,120.0,-125.0,0.455,0.556,4.3,4.0,2.21,-0.2,-0.5,0.090,0.464,0.536,2.08,-3.52,0.2,0.4,0.47,0.50,35.26,5.37,0.25,0.05,3.83,6.0
21,Chet Holmgren,AST,1.5,21.16,29.55,34.60,0.00,1.19,4.29,0.395,0.605,AST,PrizePicks,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-105.0,-114.0,0.512,0.533,1.4,1.0,1.65,-0.1,-0.5,0.061,0.476,0.524,-7.07,-1.64,0.6,0.4,0.33,0.49,27.66,5.61,0.21,0.05,1.00,6.0
35,Duncan Robinson,REB,2.5,21.75,29.95,35.43,0.81,3.45,7.74,0.610,0.390,REB,DraftKings Pick6,Orlando Magic,-8.5,202.0,113.6,13.0,100.56,14.0,105.0,-120.0,0.488,0.545,2.3,2.0,2.00,-0.2,-0.5,0.100,0.460,0.540,-5.70,-1.00,0.6,0.4,0.40,0.45,27.13,4.38,0.16,0.04,2.62,13.0
122,Quentin Grimes,PTS,6.5,14.36,21.39,29.27,1.99,8.21,20.02,0.546,0.454,PTS,Betr DFS,New York Knicks,7.5,212.5,112.3,7.0,97.71,25.0,100.0,-104.0,0.500,0.510,9.6,6.5,7.31,3.1,0.0,-0.424,0.664,0.336,32.80,-34.09,0.4,0.5,0.60,0.76,22.42,4.14,0.17,0.07,11.50,6.0


### Get top EVs for 2 legs

In [16]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 109  |  Pairs: 899  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 49  |  Pairs: 149  |  Slate: 7  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 29  |  Pairs: 95  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [19]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 106  |  Pairs: 825  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 109  |  Triples: 22721  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 49  |  Triples: 1693  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 106  |  Triples: 20034  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [23]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 29  |  Triples: 442  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
